In [ ]:
# run this once to install the required packages / Do it once!
%pip install pandas==2.3.3
%pip install numpy
%pip install matplotlib
%pip install statsmodels
%pip install scipy
%pip install openpyxl

# DynaTMT (Current version: 2.9.4 - 2026-03-19)
%pip install --upgrade "git+https://github.com/science64/DynaTMT.git"

# PBLMM (Current version: 2.1.3 - 2026-02-02)
%pip install --upgrade "git+https://github.com/science64/PBLMM.git"

In [1]:
# import the required packages / you need to run all the time!
from datetime import date
import warnings
import DynaTMT as mePROD         # should be ==> v2.9.4
import pandas as pd              # should be ==> v2.3.3
import PBLMM                     # should be ==> v2.1.3

warnings.filterwarnings("ignore")

# Check versions
print("╔════════════════════════════════════════╗")
print("║       Package Versions Loaded          ║")
print("╠════════════════════════════════════════╣")
print(f"║  DynaTMT:        v{mePROD.__version__:<20} ║")
print(f"║  Pandas:         v{pd.__version__:<20} ║")
print(f"║  PBLMM:          v{PBLMM.__version__:<20} ║")
print("╚════════════════════════════════════════╝")


╔════════════════════════════════════════╗
║       Package Versions Loaded          ║
╠════════════════════════════════════════╣
║  DynaTMT:        v2.9.4                ║
║  Pandas:         v2.3.3                ║
║  PBLMM:          v2.1.3                ║
╚════════════════════════════════════════╝


In [2]:
wd = "Example data/MS2_data" # you can define your folder here etc: C://Users/User/Data/fractionation/

nameOfStudy = "2h_CCCP_ISRIB_SB" # please define a name for your study

dataName = "20200724_SB_CCCP_ISRIB_Import_PSMs.txt" # please define the name of your data file (PSMs) here (INPUT)

conditions = ['Light', 'DMSO', 'DMSO', 'DMSO', 'CCCP', 'CCCP', 'CCCP', 'CCCP_ISRIB', 'CCCP_ISRIB', 'CCCP_ISRIB'] # define the conditions of TMT multiplexing here 

pairs = [['CCCP', 'DMSO'], ['CCCP_ISRIB', 'DMSO'], ['CCCP_ISRIB', 'CCCP']] # define the pairs of conditions you want to compare here. result will be log2(CCCP/DMSO)

In [3]:
psms = pd.read_csv(f'{wd}/{dataName}', sep='\t', header=0) # TEXT or CSV file: you provide your .txt PSM or peptide file here.
booster_removed = psms.drop('Abundance: 131C', axis = True) # remove the booster channel if present

In [4]:
process = mePROD.PD_input(booster_removed) # initiate your date here with PD_input class, if your data name is 'booster_removed'

filter_data = process.filter_PSMs(booster_removed) # filter contamination, NA samples, shared peptides

IT_adjusted = process.IT_adjustment(filter_data) # IT adjusment helpful for MS2 samples

sumNorm = process.total_intensity_normalisation(IT_adjusted) # for total intenstiy normalization

heavy = process.extract_heavy(sumNorm) # extract heavy PSMs/peptides

# light = process.extract_light(sumNorm) # extract light PSMs/peptides (OPTIONAL) ==> You need to export it later with peptide_to_protein conversion

peptide_data = process.baseline_correction(heavy, threshold=15, i_baseline=0, random=True) # baseline correction of heavy PSMs/peptides

# PBLMM analysis ==> this is the main part of the statistical analysis based on peptide based linear mixed model (LMM)
hypothesis_testing = PBLMM.HypothesisTesting()
resultFinal = hypothesis_testing.peptide_based_lmm(peptide_data,conditions=conditions,pairs=pairs)
resultFinal.reset_index(inplace=True)
resultFinal.rename(columns={'index': 'Accession'}, inplace=True)

resultFinal.to_excel(f'{wd}/{nameOfStudy}_mePROD_PBLMM_MS2_{date.today().strftime("%d.%m.%Y")}.xlsx', index=False, engine='openpyxl')

print('[#] COMPLETED: resultFinal: %s proteins x %s columns' % (resultFinal.shape[0], resultFinal.shape[1]))

Calling function: filter_PSMs
Calling function: IT_adjustment
Calling function: total_intensity_normalisation
Calling function: extract_heavy
Extraction Done Extracted Heavy PSMs/Peptides: 65547
Calling function: baseline_correction
[#] Decision of this file is: PSMs
Calling function: PSMs_to_Peptide
Calculate Protein quantifications from Peptides
Combination done
Total Number of Datapoints:  304670
['CCCP', 'DMSO'] is analysing via PBLMM...
['CCCP_ISRIB', 'DMSO'] is analysing via PBLMM...
['CCCP_ISRIB', 'CCCP'] is analysing via PBLMM...
[#] COMPLETED: resultFinal: 5060 proteins x 20 columns
